# Deliverable 2 – Campus Coffee Shop
INF201 · Sigurd Aleksander Ruud and Shem Abdul-Rahman

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
#imports and gets all the packages needed for the code to run
DATA_DIR = Path("data")
OUT_DIR = Path("output")
OUT_DIR.mkdir(exist_ok=True)

## Task 1.1 Robust file reader
Try UTF-8 first. If it fails, fall back to ISO-8859-1.

In [2]:
def read_file(path) -> tuple[str, str]:
    """Read a text file as UTF-8, fall back to ISO-8859-1. Returns (text, encoding)."""
    path = Path(path)
    try:
        return path.read_text(encoding="utf-8"), "utf-8"
    except UnicodeDecodeError:
        return path.read_text(encoding="iso-8859-1"), "iso-8859-1"

## Task 1.2 Log parser
Regex captures the 6 fields. Noise lines don't match and are skipped.

In [3]:
LINE_PATTERN = re.compile(
    r"\[(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]"   # timestamp
    r" \[(\w+)\]"                                  # register_id
    r" item=(\w+)"                                 # item
    r" price=(\d+\.\d+)"                           # price
    r" ([A-Z]{3})"                                 # currency
    r" \((SUCCESS|REFUND|CANCELLED)\)"             # status
)


def parse_log(text: str) -> list[dict]:
    """Return one dict per valid sales line; noise lines are ignored."""
    records = []
    for line in text.splitlines():
        match = LINE_PATTERN.search(line)
        if match:
            ts, reg, item, price, cur, status = match.groups()
            records.append({
                "timestamp": ts,
                "register_id": reg,
                "item": item,
                "price": float(price),
                "currency": cur,
                "status": status,
            })
    return records

In [4]:
test = """[2026-10-15 08:30:00] [POS_01] item=Latte price=4.50 EUR (SUCCESS)
--- PRINTER PAPER LOW ---"""
parse_log(test)

[{'timestamp': '2026-10-15 08:30:00',
  'register_id': 'POS_01',
  'item': 'Latte',
  'price': 4.5,
  'currency': 'EUR',
  'status': 'SUCCESS'}]

## Task 2.1 Recursive file search
The function calls itself on every sub-folder (no glob/rglob/os.walk).

In [ ]:
def find_log_files(folder, extension: str = ".log") -> list[Path]:
    """Recursively find all files with the given extension."""
    folder = Path(folder)
    found = []
    for item in folder.iterdir():
        if item.is_dir():
            found.extend(find_log_files(item, extension))   # calls itself
        elif item.is_file() and item.suffix == extension:
            found.append(item)
    return found